In [2]:
%pip install relbench scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import time

import numpy as np
import pandas as pd
import pooch
import pyarrow as pa
import pyarrow.json
import re

from relbench.base import Database, Dataset, Table
from relbench.datasets import get_dataset, get_dataset_names, register_dataset
from relbench.utils import unzip_processor

In [ ]:
DATA_DIR = Path('CNES/')
ENCODING = 'latin-1'
TABLE_INFO = {
    'tbEstabelecimento': {'pkey': 'CO_UNIDADE', 'time_col': 'DT_ATUALIZACAO'},
    'rlEstabEquipamento': {'pkey': None, 'fkeys': {'CO_UNIDADE': 'tbEstabelecimento'}, 'time_col': 'DT_ATUALIZACAO_ESTAB'},
    'tbCargaHorariaSus': {'pkey': None, 'fkeys': {'CO_UNIDADE': 'tbEstabelecimento', 'CO_PROFISSIONAL_SUS': 'tbDadosProfissionalSus'}, 'time_col': 'DT_ATUALIZACAO'},
    'tbDadosProfissionalSus': {'pkey': 'CO_PROFISSIONAL_SUS', 'time_col': 'DT_ATUALIZACAO'},
    'rlEstabServClass': {'pkey': None, 'fkeys': {'CO_UNIDADE': 'tbEstabelecimento'}, 'time_col': 'DT_ATUALIZACAO_ESTAB'},
    'rlEstabProgFundo': {'pkey': None, 'fkeys': {'CO_UNIDADE': 'tbEstabelecimento'}, 'time_col': 'DT_ATUALIZACAO_ESTAB'},
    'rlEstabAtendPrestConv': {'pkey': None, 'fkeys': {'CO_UNIDADE': 'tbEstabelecimento'}, 'time_col': 'DT_ATUALIZACAO_ESTAB'},
    'tbEquipe': {'pkey': ['CO_MUNICIPIO', 'CO_AREA', 'SEQ_EQUIPE'], 'fkeys': {'CO_UNIDADE': 'tbEstabelecimento'}, 'time_col': 'DT_ATUALIZACAO'},
}

class CnesDatasetDelta(Dataset):
    """
    Dataset do CNES que constrói o banco de dados histórico descartando
    linhas inalteradas entre os snapshots mensais.
    """
    test_timestamp = pd.Timestamp("2025-05-01")
    val_timestamp = pd.Timestamp("2025-06-01")
    test_timestamp = pd.Timestamp("2025-07-01")

    def make_db(self) -> Database:
        """
        Processa os arquivos CSV locais do CNES de forma incremental.
        """
        
        # ==========================================================================
        # 1. Identificar e ordenar todos os períodos disponíveis
        # ==========================================================================
        all_files = list(DATA_DIR.glob('*.csv'))
        period_pattern = re.compile(r'(\d{6})')
        periods = sorted(list(set(period_pattern.search(f.name).group(1) for f in all_files if period_pattern.search(f.name))))
        
        if not periods:
            raise FileNotFoundError("Nenhum arquivo CSV com período (YYYYMM) encontrado no diretório CNES/.")
        
        print(f"Períodos encontrados: {periods}")

        # ==========================================================================
        # 2. Carregar o primeiro período como base e depois adicionar apenas os deltas
        # ==========================================================================
        dfs_full = {}
        dfs_previous = {}

        for i, period in enumerate(periods):
            print(f"\nProcessando período: {period}...")
            
            # Carrega o período atual
            dfs_current = {
                name: pd.read_csv(DATA_DIR / f"{name}{period}.csv", sep=';', encoding=ENCODING, low_memory=False)
                for name in TABLE_INFO.keys() if (DATA_DIR / f"{name}{period}.csv").exists()
            }
            
            if i == 0:
                # O primeiro período é a nossa base de dados inicial
                print("  - Período base. Carregando todas as linhas.")
                dfs_full = dfs_current.copy()
            else:
                # Para períodos subsequentes, calculamos e anexamos apenas o delta
                print("  - Calculando e anexando deltas (linhas novas/alteradas).")
                for name, df_current in dfs_current.items():
                    if name not in dfs_previous or dfs_previous[name].empty:
                        # Se a tabela não existia antes, tudo é novo
                        dfs_full[name] = pd.concat([dfs_full.get(name), df_current], ignore_index=True)
                        continue

                    df_prev = dfs_previous[name]
                    
                    # Usa merge para encontrar linhas em df_current que não estão em df_prev
                    merged = pd.merge(df_prev, df_current, how='right', indicator=True)
                    delta_df = merged[merged['_merge'] == 'right_only'].drop(columns=['_merge'])
                    
                    if not delta_df.empty:
                        # Anexa apenas as linhas novas/alteradas
                        dfs_full[name] = pd.concat([dfs_full[name], delta_df], ignore_index=True)
                        print(f"    - Tabela '{name}': {len(delta_df)} linhas novas/alteradas adicionadas.")

            dfs_previous = dfs_current.copy()

        # ==========================================================================
        # 3. Pré-processamento, Propagação de Timestamps e Criação dos Objetos Table
        # (Esta parte é a mesma da versão anterior, mas agora opera nos DFs consolidados)
        # ==========================================================================
        print("\nIniciando pré-processamento final e propagação de timestamps...")
        
        COLS_TO_DROP = ['CO_USUARIO', 'DT_ATUALIZACAO_ORIGEM', 'DT_CMTP_INICIO', 'DT_CMTP_FIM', 'NU_SEQ_PROCESSO']
        for name, df in dfs_full.items():
            cols_to_drop_existing = [col for col in COLS_TO_DROP if col in df.columns]
            if cols_to_drop_existing: df.drop(columns=cols_to_drop_existing, inplace=True)

        for name, df in dfs_full.items():
            time_col = TABLE_INFO[name].get('time_col')
            if time_col and 'DT_ATUALIZACAO' in time_col: # Apenas colunas de atualização real
                for col in df.columns:
                    if 'DT_' in col:
                        dfs_full[name][col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
        
        if 'tbEstabelecimento' in dfs_full:
            df_estab_dates = dfs_full['tbEstabelecimento'][['CO_UNIDADE', 'DT_ATUALIZACAO']].dropna().drop_duplicates(subset=['CO_UNIDADE'], keep='last')
            df_estab_dates.rename(columns={'DT_ATUALIZACAO': 'DT_ATUALIZACAO_ESTAB'}, inplace=True)
            for name in ['rlEstabEquipamento', 'rlEstabServClass', 'rlEstabProgFundo', 'rlEstabAtendPrestConv']:
                if name in dfs_full:
                    dfs_full[name] = pd.merge(dfs_full[name], df_estab_dates, on='CO_UNIDADE', how='left')

        tables = {}
        for name, df in dfs_full.items():
            info = TABLE_INFO[name]
            time_col = info['time_col']
            if time_col not in df.columns: continue
            df.dropna(subset=[time_col], inplace=True)
            tables[name] = Table(df=df.reset_index(drop=True), fkey_col_to_pkey_table=info.get('fkeys', {}), pkey_col=info.get('pkey'), time_col=time_col)
            print(f"📦 Objeto Table para '{name}' criado.")
            
        return Database(tables)

# --- Exemplo de Uso ---
print("Instanciando CnesDatasetDelta...")
cnes_delta_dataset = CnesDatasetDelta()

print("\nExecutando make_db() para criar o objeto Database incremental...")
db_delta = cnes_delta_dataset.make_db()

if db_delta:
    print("\n✅ Processo concluído! O objeto 'db_delta' (incremental) está pronto.")
    
    # Vamos verificar o tamanho final da tabela de estabelecimentos
    df_estab_final = db_delta.table_dict['tbEstabelecimento'].df
    print(f"Tamanho final da tabela 'tbEstabelecimento': {len(df_estab_final)} linhas.")
    
    # Compare isso com o tamanho da tabela do último período para ver a diferença
    df_estab_ultimo_periodo = pd.read_csv(DATA_DIR / f"tbEstabelecimento{periods[-1]}.csv", sep=';', encoding=ENCODING, low_memory=False)
    print(f"Tamanho da tabela no último período ({periods[-1]}): {len(df_estab_ultimo_periodo)} linhas.")
    print("\n(O tamanho final é maior que o do último período pois acumula alterações históricas, mas menor que a soma de todos os arquivos)")

Instanciando CnesDataset...

Executando make_db() para criar o objeto Database...
✅ Tabela 'tbEstabelecimento' carregada e concatenada (total de 2062807 linhas).
✅ Tabela 'rlEstabEquipamento' carregada e concatenada (total de 4580625 linhas).
